In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import col, from_json, window, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType

sparkConf = SparkConf()
sparkConf.setMaster("spark://spark-master:7077")
sparkConf.setAppName("ShelterStreamPipeline")
sparkConf.set("spark.driver.memory", "2g")
sparkConf.set("spark.executor.cores", "1")
sparkConf.set("spark.driver.cores", "1")
sparkConf.set("spark.cores.max", "1")
sparkConf.set("spark.sql.shuffle.partitions", "2")


spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()

bucket = "de2026-group10-ass2-data"
project_id = "joekiepoekie-de"
spark.conf.set("temporaryGcsBucket", bucket)

conf = spark.sparkContext._jsc.hadoopConfiguration()
conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")

intake_schema = StructType([
    StructField("animal_id", StringType(), True),
    StructField("animal_type", StringType(), True),
    StructField("intake_type", StringType(), True),
    StructField("intake_condition", StringType(), True),
    StructField("event_time", StringType(), True)
])

# Read stream from Kafka
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "shelter-intakes") \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON
parsed_stream = raw_stream.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json(col("json_str"), intake_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_timestamp", to_timestamp(col("event_time")))

# Watermarking + 1-minute Tumbling Window Aggregation
windowed_counts = parsed_stream \
    .withWatermark("event_timestamp", "2 minutes") \
    .groupBy(
        window(col("event_timestamp"), "1 minute"),
        col("animal_type"),
        col("intake_condition")
    ).count() \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("animal_type"),
        col("intake_condition"),
        col("count").alias("intake_count")
    )

# Write each micro-batch to BigQuery using foreachBatch
def write_to_bigquery(batch_df, batch_id):
    row_count = batch_df.count()
    print(f"=== Batch {batch_id}: {row_count} rows ===")
    if row_count > 0:
        batch_df.write.format("bigquery") \
            .option("table", f"{project_id}.shelter_analytics.stream_live_intakes") \
            .mode("overwrite") \
            .save()

query = windowed_counts.writeStream \
    .outputMode("complete") \
    .trigger(processingTime="15 seconds") \
    .option("checkpointLocation", "/home/jovyan/checkpoint/shelter_stream_v2") \
    .foreachBatch(write_to_bigquery) \
    .start()

query.awaitTermination()

=== Batch 0: 0 rows ===
=== Batch 1: 8 rows ===
=== Batch 2: 11 rows ===
=== Batch 3: 12 rows ===


StreamingQueryException: [STREAM_FAILED] Query [id = 54a4c722-3bbe-4ce0-b6e1-22ff4a3898e2, runId = 1bd6b70c-25c2-4654-ac12-50cd3d0298e5] terminated with exception: Failed to create new KafkaAdminClient SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 54a4c722-3bbe-4ce0-b6e1-22ff4a3898e2, runId = 1bd6b70c-25c2-4654-ac12-50cd3d0298e5]
Current Committed Offsets: {KafkaV2[Subscribe[shelter-intakes]]: {"shelter-intakes":{"0":4622}}}
Current Available Offsets: {KafkaV2[Subscribe[shelter-intakes]]: {"shelter-intakes":{"0":4622}}}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 ForeachBatchSink, 54a4c722-3bbe-4ce0-b6e1-22ff4a3898e2, [checkpointLocation=/home/jovyan/checkpoint/shelter_stream_v2], Complete
+- ~Project [window#30-T120000ms.start AS window_start#31, window#30-T120000ms.end AS window_end#32, animal_type#17, intake_condition#19, count#22L AS intake_count#33L]
   +- ~Aggregate [window#30-T120000ms, animal_type#17, intake_condition#